# TURN-Mapper — train the learned segmenter (Colab)

Runtime → Change runtime type → **T4 GPU**. Total time ≈ 3–5 h for pre-training (it can be split across sessions: checkpoints go to Google Drive).

Steps: 1) install · 2) download CubiCasa5K · 3) convert to masks · 4) pre-train · 5) evaluate · 6) fine-tune on our own annotated plans.

CubiCasa5K is CC BY-NC 4.0 (non-commercial): fine for the FYP; never commit it or the weights to git.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/turn_ml'   # data + checkpoints survive Colab disconnects here
!mkdir -p $WORK

In [ ]:
# repo (if it is private: use https://<token>@github.com/ZakiNabeel/Turn-Mapper.git)
!git clone https://github.com/ZakiNabeel/Turn-Mapper.git /content/Turn-Mapper
%cd /content/Turn-Mapper
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install -e ".[ml]"
!python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

## 1. Download CubiCasa5K (~5.5 GB zip, once; kept on Drive)

In [ ]:
import os
if not os.path.exists(f'{WORK}/cubicasa5k.zip'):
    !wget -q --show-progress -O $WORK/cubicasa5k.zip "https://zenodo.org/record/2613548/files/cubicasa5k.zip?download=1"
!unzip -q -n $WORK/cubicasa5k.zip -d /content/
!ls /content/cubicasa5k | head

## 2. Convert SVG labels → TURN class masks (once, ~15 min; cached on Drive)

Quick check first with `--limit 20`, look at a preview, then run the full set.

In [ ]:
!turn-mapper ml prepare-cubicasa /content/cubicasa5k --out /content/cc_check --limit 20
import cv2, numpy as np, glob, matplotlib.pyplot as plt
from turn_mapper.learned.annotate import colorize
p = sorted(glob.glob('/content/cc_check/masks/*.png'))[0]
m = cv2.imread(p, -1); g = cv2.imread(p.replace('masks', 'images'), 0)
plt.figure(figsize=(14, 7)); plt.subplot(1, 2, 1); plt.imshow(g, cmap='gray'); plt.subplot(1, 2, 2); plt.imshow(colorize(m, g)); plt.show()

In [ ]:
if not os.path.exists(f'{WORK}/cc_masks/train.txt'):
    !turn-mapper ml prepare-cubicasa /content/cubicasa5k --out $WORK/cc_masks
!cp -r $WORK/cc_masks /content/cc_masks   # local copy = faster data loading

## 3. Pre-train on CubiCasa5K

Best checkpoint (by validation mIoU) is written as soon as it improves. If Colab disconnects, re-run with `--init $WORK/turn_seg_cubicasa.pt`.

In [ ]:
!turn-mapper ml train --data /content/cc_masks --out $WORK/turn_seg_cubicasa.pt \
    --epochs 30 --batch 8 --crop 512 --lr 3e-4 --workers 2

## 4. Evaluate (CubiCasa test split, per-class IoU)

In [ ]:
!turn-mapper ml eval --data /content/cc_masks --weights $WORK/turn_seg_cubicasa.pt --split test

## 5. Fine-tune on our own plans

1. On a laptop: `turn-mapper ml bootstrap-masks samples/private/fast_blocks.pdf --venue configs/venues/fast_isb.yaml --floors C-F3,C-F4,B-GF --out data/fast_masks` → correct `masks/` (see docs/ML_GUIDE.md), write `train.txt` / `val.txt`.
2. Upload `data/fast_masks` to `$WORK/fast_masks` (Drive). Keep A block out (held-out test building).

In [ ]:
!turn-mapper ml train --data $WORK/fast_masks \
    --init $WORK/turn_seg_cubicasa.pt --out $WORK/turn_seg_fast.pt --epochs 15 --lr 5e-5

## 6. Use it

Download `turn_seg_fast.pt` (+ `.json` manifest) into `weights/turn_seg.pt` locally, then:

```
turn-mapper run plan.pdf --segmenter learned --out runs/learned_try
```